In [58]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_holistic = mp.solutions.holistic


In [59]:
def drawLandmarksOnImage(modelOutput, image):
    ###################
    # Render detections on image
    ###################
    #draws face contours
    # mp_drawing.draw_landmarks(
    #     image,
    #     modelOutput.face_landmarks,
    #     mp_holistic.FACEMESH_CONTOURS,
    #     landmark_drawing_spec=None,
    #     connection_drawing_spec=mp_drawing_styles
    #     .get_default_face_mesh_contours_style(0))# there are currently 2 styles for .get_default_face_mesh_contours_style(), integer argument determines which is used 
    # #draws face tesselation # I have it turned off because I think its ugly with the default style
    # mp_drawing.draw_landmarks(
    #     image,
    #     modelOutput.face_landmarks,
    #     mp_holistic.FACEMESH_TESSELATION,
    #     landmark_drawing_spec=mp_drawing_styles
    #     .get_default_face_mesh_tesselation_style())
    #draws left hand
    mp_drawing.draw_landmarks(
        image,
        modelOutput.left_hand_landmarks,
        mp_holistic.HAND_CONNECTIONS,
        landmark_drawing_spec=mp_drawing_styles
        .get_default_hand_landmarks_style()
        ,connection_drawing_spec=mp_drawing_styles
        .get_default_hand_connections_style()
        ) 
    #draws right hand 
    mp_drawing.draw_landmarks(
        image,
        modelOutput.right_hand_landmarks,
        mp_holistic.HAND_CONNECTIONS,
        landmark_drawing_spec=mp_drawing_styles
        .get_default_hand_landmarks_style(),
        connection_drawing_spec=mp_drawing_styles
        .get_default_hand_connections_style())
    #draws pose
    mp_drawing.draw_landmarks(
        image,
        modelOutput.pose_landmarks,
        mp_holistic.POSE_CONNECTIONS,
        landmark_drawing_spec=mp_drawing_styles
        .get_default_pose_landmarks_style()) 

In [60]:
def getLandmarkOutput(frame, landmarkSolution):
    image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    image.flags.writeable = False
    
    # Make detection
    landmarkOutput = landmarkSolution.process(image)
    #from MediaPipe documentation
    """
    Returns:
    A NamedTuple with fields describing the landmarks on the most prominate
    person detected:
    1) "pose_landmarks" field that contains the pose landmarks.
    2) "pose_world_landmarks" field that contains the pose landmarks in
    real-world 3D coordinates that are in meters with the origin at the
    center between hips.
    3) "left_hand_landmarks" field that contains the left-hand landmarks.
    4) "right_hand_landmarks" field that contains the right-hand landmarks.
    5) "face_landmarks" field that contains the face landmarks.
    6) "segmentation_mask" field that contains the segmentation mask if
        "enable_segmentation" is set to true.
        
        
    Enum values for pose_landmarks and pose_world_landmarks are the same
    """

    # Recolor back to BGR, openCV is weird like that
    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    
    return landmarkOutput, image

In [61]:
def modelOutputToLandmarkDict(modelOutput) -> dict[str,dict[object,object]]:
    
    poseLandmarks = modelOutput.pose_landmarks.landmark if modelOutput.pose_landmarks is not None else None
    poseWorldLandmarks = modelOutput.pose_world_landmarks.landmark if modelOutput.pose_world_landmarks is not None else None
    leftHandLandmarks = modelOutput.left_hand_landmarks.landmark if modelOutput.left_hand_landmarks is not None else None
    rightHandLandmarks = modelOutput.right_hand_landmarks.landmark if modelOutput.right_hand_landmarks is not None else None
    
    poseEnum = mp.solutions.pose.PoseLandmark #enum/name-values of pose landmarks in mediaPipe
    handEnum = mp.solutions.hands.HandLandmark #enum/name-values of hand landmarks in mediaPipe
    
    #tuple for what we want to include in our data
    landmarkTypesWithEnums = (("poseLandmarks", poseLandmarks,poseEnum),("poseWorldLandmarks",poseWorldLandmarks,poseEnum),
                                ("leftHandLandmarks",leftHandLandmarks,handEnum),("rightHandLandmarks",rightHandLandmarks,handEnum))
    
    #logic for turning data we want into a unified customly formated dictionary, instead of an unnamed tuple
    formatedLandmarks:dict[str,dict[object,object]] = dict()
    formatedLandmarkType:dict[object,object] = dict()
    for pair in landmarkTypesWithEnums:
        landmarkTypeName,landmarkType, enum = pair
        
        if landmarkType is not None: # have to check to make sure landmark was defined
            landmarkTypeIsPresent = 1
            formatedLandmarkType.update({'present':landmarkTypeIsPresent})
            
            for enumerate in enum:
                
                #.value is the value of the enum For example it is 1 in the enum NUM = 1
                if landmarkType[enumerate.value] is not None: 
                    landmarkX = landmarkType[enumerate.value].x
                    landmarkY = landmarkType[enumerate.value].y
                    landmarkZ = landmarkType[enumerate.value].z
                    landmarkPresent = 1
                else: # cordinate is normalizes to be in range 0..1 so we can use -1 to signify lack of presence in model
                    landmarkX = -1
                    landmarkY = -1
                    landmarkZ = -1
                    landmarkPresent = 0
                
                #.name is the name of the enum For example it is "NUM" in the enum NUM = 1
                landmark = {enumerate.name:{'x':landmarkX,'y':landmarkY,'z':landmarkZ,'present':landmarkPresent}}
                    
                formatedLandmarkType.update(landmark)
                
        else: #landmark type wasn't found so set all landmarks in type to not present and default
            landmarkTypeIsPresent = 0
            formatedLandmarkType.update({'present':landmarkTypeIsPresent})
            
            for enumerate in enum:
                landmarkX = -1
                landmarkY = -1
                landmarkZ = -1
                landmarkPresent = 0
                
                #.name is the name of the enum For example it is "NUM" in the enum NUM = 1
                landmark = {enumerate.name:{'x':landmarkX,'y':landmarkY,'z':landmarkZ,'present':landmarkPresent}}
                    
                formatedLandmarkType.update(landmark)
                
        formatedLandmarks.update({landmarkTypeName:formatedLandmarkType})
        formatedLandmarkType = dict() #clear dict for next itration
    
    return formatedLandmarks

In [62]:
def appendFrameModelOutput(modelOutput, data=dict()):    
    poseLandmarks = modelOutput.pose_landmarks.landmark if modelOutput.pose_landmarks is not None else None
    poseWorldLandmarks = modelOutput.pose_world_landmarks.landmark if modelOutput.pose_world_landmarks is not None else None
    leftHandLandmarks = modelOutput.left_hand_landmarks.landmark if modelOutput.left_hand_landmarks is not None else None
    rightHandLandmarks = modelOutput.right_hand_landmarks.landmark if modelOutput.right_hand_landmarks is not None else None
    
    poseEnum = mp.solutions.pose.PoseLandmark #enum/name-values of pose landmarks in mediaPipe
    handEnum = mp.solutions.hands.HandLandmark #enum/name-values of hand landmarks in mediaPipe
    
    #tuple for what we want to include in our data
    landmarkTypesWithEnums = (("poseLandmarks", poseLandmarks,poseEnum),("poseWorldLandmarks",poseWorldLandmarks,poseEnum),
                                ("leftHandLandmarks",leftHandLandmarks,handEnum),("rightHandLandmarks",rightHandLandmarks,handEnum))
    
    #logic for turning data we want into a unified customly formated dictionary, instead of an unnamed tuple
            
    for pair in landmarkTypesWithEnums:
        landmarkTypeName,landmarkType, enum = pair
        
        if landmarkType is not None: # have to check to make sure landmark was defined
            key = landmarkTypeName + '_present'
            data.update({key: data.setdefault(key, []) + [1]}) #1 for present 0 for not present
            
            for enumerate in enum:
                
                #.value is the value of the enum For example it is 1 in the enum NUM = 1
                if landmarkType[enumerate.value] is not None: 
                    landmarkX = landmarkType[enumerate.value].x
                    landmarkY = landmarkType[enumerate.value].y
                    landmarkZ = landmarkType[enumerate.value].z
                    landmarkPresent = 1
                else: # cordinate is normalizes to be in range 0..1 so we can use -1 to signify lack of presence in model
                    landmarkX = -1
                    landmarkY = -1
                    landmarkZ = -1
                    landmarkPresent = 0
                
                #.name is the name of the enum For example it is "NUM" in the enum NUM = 1
                key = landmarkTypeName + "_" + enumerate.name + '_x'
                data.update({key: data.setdefault(key, []) + [landmarkX]})
                key = landmarkTypeName + "_" + enumerate.name + '_y'
                data.update({key: data.setdefault(key, []) + [landmarkY]})
                key = landmarkTypeName + "_" + enumerate.name + '_z'
                data.update({key: data.setdefault(key, []) + [landmarkZ]})
                key = landmarkTypeName + "_" + enumerate.name + '_present'
                data.update({key: data.setdefault(key, []) + [landmarkPresent]})
                                
        else: #landmark type wasn't found so set all landmarks in type to not present and default
            key = landmarkTypeName + '_present'
            data.update({key: data.setdefault(key, []) + [0]}) #1 for present 0 for not present
            
            for enumerate in enum:
                landmarkX = -1
                landmarkY = -1
                landmarkZ = -1
                landmarkPresent = 0
                
                key = landmarkTypeName + "_" + enumerate.name + '_x'
                data.update({key: data.setdefault(key, []) + [landmarkX]})
                key = landmarkTypeName + "_" + enumerate.name + '_y'
                data.update({key: data.setdefault(key, []) + [landmarkY]})
                key = landmarkTypeName + "_" + enumerate.name + '_z'
                data.update({key: data.setdefault(key, []) + [landmarkZ]})
                key = landmarkTypeName + "_" + enumerate.name + '_present'
                data.update({key: data.setdefault(key, []) + [landmarkPresent]})
                                        
    return data

In [63]:
cap = cv2.VideoCapture("archive (2)/ASL_Citizen/videos/4.7299129501965353e-7-seedSOUR.mp4")
#can you change the capture rate to be higher(this would help process mp4s faster, instead of the default 30 fps)

# Setup mediapipe holistic solution instance
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    data = dict()
    while cap.isOpened():
        #get input
        success, frame = cap.read()
        
        #error handling
        if not success:
            # If loading a video, use 'break' if using live feed, use 'continue'.
            # continue
            break
        
        modelOutput, image = getLandmarkOutput(frame, holistic)
        tmp = modelOutputToLandmarkDict(modelOutput)
        appendFrameModelOutput(modelOutput, data)
        
        #show original frame
        cv2.imshow("Original Feed", frame)
        
        #show new image in python window
        drawLandmarksOnImage(modelOutput, image)
        cv2.imshow('Mediapipe Feed', image)
        
        #show only detections in python window
        blackScreen = np.zeros(image.shape)
        justDetections = blackScreen
        drawLandmarksOnImage(modelOutput, justDetections)
        cv2.imshow('Detections Feed', justDetections)


        keyPressed = cv2.waitKey(10) & 0xFF
        #quit if q is pressed
        if keyPressed == ord('q'):
            break
        
        #pause if p is pressed
        if keyPressed == ord('p'):
            while cap.isOpened():
                #unpause if p is pressed again
                if cv2.waitKey(10) & 0xFF == ord('p'):
                    break
        
    cap.release()
    cv2.destroyAllWindows()